# Two-component GPE — Colab launch

Use a GPU runtime and the **colab** Git branch. The GUI keeps Experiment → Launch → Monitor.

**Modification boundary:** only `two_component_GPE` may be developed or edited.
The sibling single-component `GPE_simulator` is **read-only reference material**.
Do not modify its code, Git history, notebooks, environments, configurations or outputs.
Do not point this notebook's repository or output folder at that package.

Each launch creates a new folder on Google Drive containing `config/run.yaml`,
`notes.txt`, `data/`, `monitor/`, `logs/`, and the exact runtime sources.
Saved wavefunctions are a pair (`psi1_*.npy`, `psi2_*.npy`) committed by `psi_*.json`.
Set periodic saving intervals: runtime termination loses unsaved steps and can
interrupt a write. A browser disconnect and runtime destruction are different events.
See `implementation_note.md` for the accepted imaginary-time accuracy limitations.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Change only these paths to match your Google Drive folders.
REPOSITORY = Path('/content/drive/MyDrive/two_component_GPE')
SIMULATIONS = Path('/content/drive/MyDrive/simulations')

if 'GPE_simulator' in REPOSITORY.parts or 'GPE_simulator' in SIMULATIONS.parts:
    raise ValueError('GPE_simulator is read-only. Select the two-component repository and a separate output folder.')
if not (REPOSITORY / 'implementation_note.md').is_file():
    raise ValueError('Select the two_component_GPE repository folder.')
branch = subprocess.check_output(['git', '-C', str(REPOSITORY), 'branch', '--show-current'], text=True).strip()
if branch != 'colab':
    raise ValueError(f'Current branch is {branch!r}. Save edits and switch this repository to colab before continuing.')
for package in ('gpe', 'config', 'gui_periphery'):
    module = sys.modules.get(package)
    if module is not None and getattr(module, '__file__', None):
        if not Path(module.__file__).resolve().is_relative_to(REPOSITORY.resolve()):
            raise RuntimeError('Restart the runtime before switching simulator packages; an older package is already imported.')
os.chdir(REPOSITORY)
if str(REPOSITORY) not in sys.path:
    sys.path.insert(0, str(REPOSITORY))
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'gui_periphery/requirements.txt'])


## First workflow test

Start with `config/examples/colab_smoke.yaml`. The cell below downloads it so the
GUI's **Open YAML** button can select it from your computer.

This is a 32³ workflow test with 10 imaginary steps and 20 real steps. It uses
nonzero intercomponent coupling and a drive on component 2 only. It writes scalar
history, spectra, density, phase and TOF samples every 5 steps, and five paired
snapshots (about 5 MiB total). These are test parameters, not a converged ground state
or calibrated experimental run.

After opening the GUI:

1. **Experiment → Open YAML:** select the downloaded `colab_smoke.yaml`.
2. **Launch:** name the run `colab_smoke_001`, add a short description and verify
   the Drive simulations folder. Click **Prepare run on Drive**.
3. Install CuPy if needed, then click **Launch simulation**.
4. **Monitor:** enable refresh and select each stage/component. This run may
   finish quickly; its saved history should still be visible. Both norms should
   remain close to 1 because this test has no absorber or momentum cutoff.
5. Check the named folder in the Google Drive web interface. After the run has
   completed and files have synced, reconnect with a new runtime, repeat setup,
   and **Reopen a saved run** using the same folder. Confirm its plots still load.
   Reopening displays saved results; do not prepare the same launch name again.


In [ ]:
from google.colab import files
files.download(str(REPOSITORY / 'config/examples/colab_smoke.yaml'))


## Open the interface

1. Set both populations, scattering lengths, initial widths and component potentials.
2. In **Launch**, choose the launch name and describe the question being tested.
   The simulations parent defaults to the path above. **Prepare run on Drive**
   captures the current YAML; later editor changes require a new prepared run.
3. Use the matching CUDA wheel if CuPy needs installing, then **Launch simulation**.
4. In **Monitor**, select the stage/component. After reconnecting, mount Drive,
   run these setup cells again and reopen the existing launch folder to view its data.
   Launch a new named run to continue from a complete paired snapshot manifest.

Closing the GUI does not intentionally stop a running simulation. Deleting or
losing the runtime stops computation. Completed writes target Drive throughout;
a final copy-out step is not required. Drive syncing/quota still applies.


In [ ]:
from gui_periphery.colab import show
show(output_parent=SIMULATIONS)


## GPU validation before research runs (optional cell)

Run the actual CUDA checks on this runtime before a long simulation. They do not
validate your experimental grid/timestep convergence or a 1024³ memory budget.
The tests use temporary small fields; production launch output remains on Drive.


In [ ]:
test_environment = dict(os.environ)
test_environment.pop('GPE_DRIVE_MOUNT', None)
subprocess.check_call([sys.executable, '-B', '-m', 'unittest', 'discover', '-s', 'tests',
                       '-p', 'test_propagators.py', '-v'], env=test_environment)
